# Stage 1: relevance detection

Binary classifier: is a sentence `None` or legally labelled? Evaluated on the full 1502-sentence test set.

## 1. Setup

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

TEST_DF_PATH = Path("predictions/test_df.csv")
REGISTRY_PATH = Path("roberta_models/best_models_registry.json")
MODELS_DIR = Path("roberta_models")

TEXT_COL = "sent_text"
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)

Device: cuda


## 2. Load test data

In [2]:
test_df = pd.read_csv(TEST_DF_PATH, keep_default_na=False)
test_df["y_labelled"] = (test_df["label"] != "None").astype(int)

print("Test rows:", len(test_df))
test_df["y_labelled"].value_counts()

Test rows: 1502


y_labelled
1    961
0    541
Name: count, dtype: int64

## 3. Load the Stage 1 model

In [3]:
with open(REGISTRY_PATH, "r", encoding="utf-8") as f:
    registry = json.load(f)

model_info = registry["stage1_gatekeeper"]
model_path = MODELS_DIR / Path(model_info["saved_path"]).name
labels = model_info["labels"]
max_len = model_info.get("max_len", 256)

if not model_path.exists():
    raise FileNotFoundError(f"Model path not found: {model_path}")

print("Model path:", model_path)
print("Labels (in order):", labels)

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.to(DEVICE)
model.eval()

Model path: roberta_models\stage1_gatekeeper_best_20260203_104234
Labels (in order): ['not_labelled', 'labelled']


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(40000, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

## 4. Run inference

In [4]:
@torch.no_grad()
def predict_probs(texts, tokenizer, model, max_len, batch_size=32):
    all_probs = []

    for start in tqdm(range(0, len(texts), batch_size), desc="Stage 1 inference"):
        batch = texts[start:start + batch_size]

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt"
        )
        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1)
        all_probs.append(probs.cpu().numpy())

    return np.vstack(all_probs)


texts = test_df[TEXT_COL].fillna("").astype(str).tolist()
probs = predict_probs(texts, tokenizer, model, max_len, BATCH_SIZE)

pred_idx = probs.argmax(axis=1)

probs.shape

Stage 1 inference:   0%|          | 0/47 [00:00<?, ?it/s]

(1502, 2)

## 5. Attach predictions and save

In [5]:
test_df["p_none"] = probs[:, 0]
test_df["p_labelled"] = probs[:, 1]
test_df["stage1_pred_idx"] = pred_idx
test_df["stage1_pred_label"] = np.where(pred_idx == 0, "not_labelled", "labelled")

OUTPUT_PATH = Path("predictions/stage1_relevance_inference.csv")
test_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Saved:", OUTPUT_PATH)
test_df[["sent_text", "label", "y_labelled", "stage1_pred_idx"]].head()

Saved: predictions\stage1_relevance_inference.csv


,sent_text,label,y_labelled,stage1_pred_idx
0,De kantonrechter heeft in het bestreden vonnis...,None,0,0
1,Deze feiten zijn in hoger beroep niet in gesch...,None,0,1
2,Op [datum] heeft [geïntimeerde] een bedrag van...,materiele feiten,1,1
3,Op [datum] heeft [appellante] een schriftelijk...,materiele feiten,1,1
4,Deze verklaring houdt onder meer in: “Dit bedr...,None,0,0


## 6. Evaluate

Sanity check: should reproduce the previously reported Stage 1 numbers (accuracy 0.7856, macro F1 0.7562).

In [6]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

STAGE1_LABELS = [0, 1]
STAGE1_NAMES = ["not_labelled / None", "labelled"]

y_true = test_df["y_labelled"]
y_pred = test_df["stage1_pred_idx"]

acc = accuracy_score(y_true, y_pred)
print("Stage 1 accuracy:", round(acc, 4))

print("\nClassification Report")
print(
    classification_report(
        y_true, y_pred, labels=STAGE1_LABELS, target_names=STAGE1_NAMES, digits=4, zero_division=0
    )
)

cm = confusion_matrix(y_true, y_pred, labels=STAGE1_LABELS)
pd.DataFrame(
    cm,
    index=["true_not_labelled_None", "true_labelled"],
    columns=["pred_not_labelled_None", "pred_labelled"]
)

Stage 1 accuracy: 0.7856

Classification Report
                     precision    recall  f1-score   support

not_labelled / None     0.7494    0.6081    0.6714       541
           labelled     0.8006    0.8855    0.8409       961

           accuracy                         0.7856      1502
          macro avg     0.7750    0.7468    0.7562      1502
       weighted avg     0.7821    0.7856    0.7799      1502



,pred_not_labelled_None,pred_labelled
true_not_labelled_None,329,212
true_labelled,110,851


## 7. Header-prior Bayesian fusion

Same methodology as `01_five_way_inference.ipynb` section 8: combine text probabilities with a header-conditioned prior in log-space (`log P(y|s,h) ∝ log P(y|s) + λ·log P(y|h)`), reporting both λ=1 and a validation-tuned λ.

In [7]:
TRAIN_DF_PATH = Path("predictions/train_df.csv")
EVAL_DF_PATH = Path("predictions/eval_df.csv")
ALPHA = 1.0
STAGE1_CLASSES = [0, 1]

train_df = pd.read_csv(TRAIN_DF_PATH, keep_default_na=False)
eval_df = pd.read_csv(EVAL_DF_PATH, keep_default_na=False)
train_df["y_labelled"] = (train_df["label"] != "None").astype(int)
eval_df["y_labelled"] = (eval_df["label"] != "None").astype(int)

print("Train rows:", len(train_df))
print("Validation rows:", len(eval_df))


def make_header_prior(df_train, label_col, labels, alpha=1.0):
    counts = (
        df_train.groupby(["hdr_group", label_col]).size()
        .unstack(fill_value=0)
        .reindex(columns=labels, fill_value=0)
    )
    probs = counts + alpha
    return probs.div(probs.sum(axis=1), axis=0)


def make_global_prior(df_train, label_col, labels, alpha=1.0):
    counts = df_train[label_col].value_counts().reindex(labels, fill_value=0) + alpha
    return (counts / counts.sum()).values


def get_meta_prior_df(df_apply, header_prior_train, global_prior, labels):
    def get_prior(hdr_group):
        if hdr_group in header_prior_train.index:
            return header_prior_train.loc[hdr_group].values
        return global_prior

    mat = np.vstack(df_apply["hdr_group"].apply(get_prior))
    return pd.DataFrame(mat, columns=labels, index=df_apply.index)


def combine_log_scores(text_probs, metadata_probs, lam, eps=1e-12):
    text = np.clip(text_probs.values, eps, 1.0)
    metadata = np.clip(metadata_probs.values, eps, 1.0)

    scores = np.log(text) + lam * np.log(metadata)
    scores -= scores.max(axis=1, keepdims=True)
    scores = np.exp(scores)
    scores /= scores.sum(axis=1, keepdims=True)

    return pd.DataFrame(scores, columns=text_probs.columns, index=text_probs.index)

Train rows: 5608
Validation rows: 1281


In [8]:
from sklearn.metrics import precision_recall_fscore_support

eval_texts = eval_df[TEXT_COL].fillna("").astype(str).tolist()
eval_probs = predict_probs(eval_texts, tokenizer, model, max_len, BATCH_SIZE)
eval_text_probs = pd.DataFrame(eval_probs, columns=STAGE1_CLASSES, index=eval_df.index)

header_prior_s1 = make_header_prior(train_df, "y_labelled", STAGE1_CLASSES, ALPHA)
global_prior_s1 = make_global_prior(train_df, "y_labelled", STAGE1_CLASSES, ALPHA)
eval_meta_prior = get_meta_prior_df(eval_df, header_prior_s1, global_prior_s1, STAGE1_CLASSES)

LAMBDA_GRID = [0, 0.1, 0.25, 0.5, 1, 2, 3, 5]
sweep_results = []

for lam in LAMBDA_GRID:
    fused = combine_log_scores(eval_text_probs, eval_meta_prior, lam)
    pred = fused.idxmax(axis=1)
    _, _, f1, _ = precision_recall_fscore_support(
        eval_df["y_labelled"], pred, labels=STAGE1_CLASSES, average="macro", zero_division=0
    )
    sweep_results.append({"lambda": lam, "val_macro_f1": round(f1, 4)})

sweep_df = pd.DataFrame(sweep_results)
best_lambda = sweep_df.loc[sweep_df["val_macro_f1"].idxmax(), "lambda"]

print("Validation lambda sweep:")
print(sweep_df)
print(f"\nBest lambda on validation: {best_lambda}")

Stage 1 inference:   0%|          | 0/41 [00:00<?, ?it/s]

Validation lambda sweep:
   lambda  val_macro_f1
0    0.00        0.7648
1    0.10        0.7614
2    0.25        0.7557
3    0.50        0.7439
4    1.00        0.7100
5    2.00        0.6578
6    3.00        0.6229
7    5.00        0.5191

Best lambda on validation: 0.0


In [9]:
test_text_probs = test_df[["p_none", "p_labelled"]].copy()
test_text_probs.columns = STAGE1_CLASSES

test_meta_prior = get_meta_prior_df(test_df, header_prior_s1, global_prior_s1, STAGE1_CLASSES)

results = []

for lam, name in [(0, "no fusion (lambda=0)"), (1, "lambda = 1"), (best_lambda, f"tuned lambda ({best_lambda})")]:
    fused_test = combine_log_scores(test_text_probs, test_meta_prior, lam)
    pred_test = fused_test.idxmax(axis=1)

    acc = accuracy_score(test_df["y_labelled"], pred_test)
    _, _, f1, _ = precision_recall_fscore_support(
        test_df["y_labelled"], pred_test, labels=STAGE1_CLASSES, average="macro", zero_division=0
    )
    results.append({"setting": name, "lambda": lam, "test_accuracy": round(acc, 4), "test_macro_f1": round(f1, 4)})
    test_df[f"stage1header_pred_lam_{lam}"] = pred_test

results_df = pd.DataFrame(results)
print(results_df)

                setting  lambda  test_accuracy  test_macro_f1
0  no fusion (lambda=0)     0.0         0.7856         0.7562
1            lambda = 1     1.0         0.7796         0.7396
2    tuned lambda (0.0)     0.0         0.7856         0.7562


In [10]:
print(f"Full classification report at tuned lambda={best_lambda} (test set)")
print(classification_report(
    test_df["y_labelled"], pred_test, labels=STAGE1_CLASSES,
    target_names=["not_labelled / None", "labelled"], digits=4, zero_division=0
))

Full classification report at tuned lambda=0.0 (test set)
                     precision    recall  f1-score   support

not_labelled / None     0.7494    0.6081    0.6714       541
           labelled     0.8006    0.8855    0.8409       961

           accuracy                         0.7856      1502
          macro avg     0.7750    0.7468    0.7562      1502
       weighted avg     0.7821    0.7856    0.7799      1502

